# P10-3 · Phỏng vấn

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import numpy as np
def softmax(z):
    z = np.asarray(z, float)
    e = np.exp(z - z.max(axis=-1, keepdims=True))     # trừ max: chống tràn số
    return e / e.sum(axis=-1, keepdims=True)

softmax([1000, 1001, 1002])   # [0.090, 0.245, 0.665]; bản không trừ max cho nan

array([0.09003057, 0.24472847, 0.66524096])


## Bài tập 2: Bài coding {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import heapq

def top_k(embs, q, k):
    heap = []                                   # min-heap (điểm, chỉ số), giữ k phần tử lớn nhất
    for i, e in enumerate(embs):
        s = sum(a * b for a, b in zip(e, q))
        if len(heap) < k:
            heapq.heappush(heap, (s, i))
        elif s > heap[0][0]:
            heapq.heapreplace(heap, (s, i))
    return [i for _, i in sorted(heap, reverse=True)]

print(top_k([[1, 0], [0.6, 0.8], [0, 1], [0.8, 0.6]], [1, 0], 2))   # [0, 3]

[0, 3]


## Bài tập 6: Vòng coding: metric xếp hạng có test {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
def precision_at_k(du_doan, dung, k):
    # TODO
    return 0.0

def mrr(ds_du_doan, ds_dung):
    # TODO: trung bình 1/hạng của phần tử đúng đầu tiên trong mỗi danh sách (0 nếu không có)
    return 0.0

In [ ]:
def precision_at_k(du_doan, dung, k):
    if k <= 0:
        raise ValueError("k phải dương")
    top, da_thay = [], set()
    for x in du_doan:                      # bỏ trùng, giữ thứ tự
        if x not in da_thay:
            da_thay.add(x); top.append(x)
    return sum(x in dung for x in top[:k]) / k

def mrr(ds_du_doan, ds_dung):
    tong = 0.0
    for du_doan, dung in zip(ds_du_doan, ds_dung):
        tong += next((1 / (i + 1) for i, x in enumerate(du_doan) if x in dung), 0.0)
    return tong / len(ds_du_doan) if ds_du_doan else 0.0

assert precision_at_k(["a", "b", "c"], {"a", "c"}, 2) == 0.5
assert precision_at_k(["a"], {"a"}, 3) == 1 / 3              # k lớn hơn số dự đoán
assert precision_at_k(["a", "a", "b"], {"a", "b"}, 2) == 1.0  # trùng chỉ tính một lần
assert precision_at_k(["a"], set(), 1) == 0.0                 # không có đáp án đúng
assert mrr([["x", "y"]], [{"y"}]) == 0.5
assert mrr([], []) == 0.0
print("qua hết test")

qua hết test


## Bài tập 7: Ước lượng nhanh cho bài thiết kế hệ thống {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
def uoc_luong(nguoi, cau_ngay, gio_lam=10, he_so_dinh=3, token_moi_cau=3500, usd_trieu_token=1.0,
              so_tai_lieu=50_000, doan_moi_tl=20, chieu=1024, byte_moi_so=4):
    # TODO: trả về dict {"rps_dinh", "token_ngay", "usd_thang", "index_gb"}
    return {}

In [ ]:
def uoc_luong(nguoi, cau_ngay, gio_lam=10, he_so_dinh=3, token_moi_cau=3500, usd_trieu_token=1.0,
              so_tai_lieu=50_000, doan_moi_tl=20, chieu=1024, byte_moi_so=4):
    cau = nguoi * cau_ngay
    return {"rps_dinh": cau / (gio_lam * 3600) * he_so_dinh,
            "token_ngay": cau * token_moi_cau,
            "usd_thang": cau * token_moi_cau / 1e6 * usd_trieu_token * 30,
            "index_gb": so_tai_lieu * doan_moi_tl * chieu * byte_moi_so / 1e9}

for ten, n in [("cơ bản", 20000), ("gấp 10", 200000)]:
    r = uoc_luong(n, 5)
    print(f"{ten:7s}: {r['rps_dinh']:.1f} req/s đỉnh | {r['token_ngay'] / 1e6:.0f} triệu token/ngày | "
          f"{r['usd_thang']:,.0f} USD/tháng | index {r['index_gb']:.1f} GB")

cơ bản : 8.3 req/s đỉnh | 350 triệu token/ngày | 10,500 USD/tháng | index 4.1 GB
gấp 10 : 83.3 req/s đỉnh | 3500 triệu token/ngày | 105,000 USD/tháng | index 4.1 GB
